<a href="https://colab.research.google.com/github/Elwing-Chou/tibame_20260714/blob/main/tibame_20260721.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def is_collatz_converge(n):
    # 我如何確定不收斂(準備一個很大的次數, 如果他超過次數還沒收斂, False)
    count = 0
    while True:
        # print(n)
        if n == 1:
            break
        if n % 2 == 1:
            n = n * 3 + 1
        else:
            n = n // 2
        count = count + 1
        # 如果這麼多次還沒到1, 那我覺得這n很有可能真不收斂
        if count == 999999999999999:
            return False
    return True

In [6]:
# 把上面變成一個函示(傳入一個n, 回傳n做出的序列的最大值)

def return_collatz_max(n):
    maxv = float("-inf")
    while True:
        # print(n)
        maxv = max(maxv, n)
        if n == 1:
            break
        if n % 2 == 1:
            n = n * 3 + 1
        else:
            n = n // 2
    return maxv

return_collatz_max(3)

16

1. 3 -> 10 -> 5 -> 16 -> 8 ....

2. 7 -> 22 -> 11 -> 34 -> 17 -> 52 > 26 -> 13 -> 40 -> 20 -> 10 -> 5(我之前已經做過了, 後來的最大值是16, 不可能比52大了, 所以後面都不用做了)

3. 做過的不要重複做 -> Dynamic Programming -> 記錄起來(用空間換時間)

In [12]:
# 你等等要幫我從1~100, 建立一個 原本-> 最大值 的字典
# {1:1, 2:2, 3:16, 4:4...}
# profiling: 真的數一下幾秒
# time的time()可以幫你回傳現在的unix timestamp
import time

# record: dynamic programming核心, 我要把算過的最大值記錄起來
# 如果你要運算的東西已經算過了, 那就把剛剛算過的最大值直接拿出來就好
start_t = time.time()
result = {}
n_max = 1000000
for i in range(n_max):
    n = i + 1
    ans = return_collatz_max(n)
    result[n] = ans
# print(result)
end_t = time.time()
dif_t = end_t - start_t
print("時間是:", dif_t)

時間是: 35.61597728729248


In [29]:
# 你等等要幫我從1~100, 建立一個 原本-> 最大值 的字典
# {1:1, 2:2, 3:16, 4:4...}
# profiling: 真的數一下幾秒
# time的time()可以幫你回傳現在的unix timestamp
import time

# record: dynamic programming核心, 我要把算過的最大值記錄起來
# 如果你要運算的東西已經算過了, 那就把剛剛算過的最大值直接拿出來就好
record = {}

def return_collatz_max_dp(n):
    # !
    n_copy = n
    maxv = float("-inf")
    while True:
        # ! 提早結束, 已經記錄過
        if n in record:
            maxv = max(maxv, record[n])
            break
        maxv = max(maxv, n)
        if n == 1:
            break
        if n % 2 == 1:
            n = n * 3 + 1
        else:
            n = n // 2
    # ! 開關
    record[n_copy] = maxv
    return maxv

start_t = time.time()
result = {}
n_max = 1000000
for i in range(n_max):
    n = i + 1
    ans = return_collatz_max_dp(n)
    result[n] = ans
end_t = time.time()
dif_t = end_t - start_t
print("時間是:", dif_t)
# print("紀錄:", record)
# print("結果:", result)

時間是: 3.236475706100464


1. 上面這個是已經不錯了, 你已經省了非常多時間了, 但我們會思考, 可不可以省更多, 一路上的數字都記錄

2. 標準dp: 先寫遞迴是比較簡單的

3. 什麼是遞迴?

In [ ]:
# 迴圈版本
total = 0
for i in range(10):
    total = total + (i + 1)
print(total)

# 遞迴: 我只解決一小部分, 把其他部分丟給別人解決
# 1+2+3+...+10-> 我把2+10丟給別人解決, 我自己只做1 + 他的答案
# 2+...+10 -> 我把3+10丟給別人解決, 我自己只做2 + 他的答案
# 老師叫我回答(1+...+10), 我跟老師說, 你請另外同學做(2+...+10), 等他有答案了, 我再補1+
# 如果可以無限偷懶, 不會結束, 所以一定要有個不偷懶的人(終止條件)
def add(start, end):
    print("這位同學被要求做", start, "加到", end)
    # 終止條件
    if start == end:
        print("不偷懶說出答案是", end)
        return end
    # 偷懶的人(中文說)
    else:
        print("要偷懶, 找另外一個同學做", start+1, "加到", end)
        other = add(start+1, end)
        print("那位同學告訴我答案是", other)
        ans = start + other
        print("我補上", start, "回答答案是", ans)
        return ans

add(1, 5)

In [36]:
# 遞迴: 我只解決一小部分, 把其他部分丟給別人解決
# 1+2+3+...+10-> 我把2+10丟給別人解決, 我自己只做1 + 他的答案
# 2+...+10 -> 我把3+10丟給別人解決, 我自己只做2 + 他的答案
# 老師叫我回答(1+...+10), 我跟老師說, 你請另外同學做(2+...+10), 等他有答案了, 我再補1+
# 如果可以無限偷懶, 不會結束, 所以一定要有個不偷懶的人(終止條件)
def add(start, end, serial):
    print("同學", serial, "被要求做", start, "加到", end)
    # 終止條件
    if start == end:
        print("不偷懶說出答案是", end)
        return end
    # 偷懶的人(中文說)
    else:
        other_serial = serial + 1
        print("要偷懶, 找另外一個同學", other_serial, "做", start+1, "加到", end)
        other = add(start+1, end, other_serial)
        print("同學", other_serial, "告訴我答案是", other)
        ans = start + other
        print("同學", serial, "補上", start, "回答答案是", ans)
        return ans

add(1, 3, serial=0)

同學 0 被要求做 1 加到 3
要偷懶, 找另外一個同學 1 做 2 加到 3
同學 1 被要求做 2 加到 3
要偷懶, 找另外一個同學 2 做 3 加到 3
同學 2 被要求做 3 加到 3
不偷懶說出答案是 3
同學 2 告訴我答案是 3
同學 1 補上 2 回答答案是 5
同學 1 告訴我答案是 5
同學 0 補上 1 回答答案是 6


6